In [ ]:
from openai import base_url

"""
LangChain 1.0 - 自定义工具 (@tool 装饰器)
=========================================
"""

In [1]:
import os
import sys

In [2]:
# 添加tools目录到路径
tools_dir = r'D:\code\LangChain-lesson\My-test\notebook\基础\tools'
sys.path.insert(0, tools_dir)

In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
# 导入自定义工具
from weather import get_weather
from calculator import calculator
from web_search import web_search
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
if not api_key or api_key == "your_dashscope_api_key_here":
    raise ValueError(
        "\n请先在 .env 文件中设置有效的 GROQ_API_KEY\n"
        "访问 https://console.groq.com/keys 获取免费密钥"
    )
model = init_chat_model(
    model = "qwen-max",
    model_provider="openai",
    api_key = api_key,
    base_url = base_url,
)

# 1.获取当前时间工具

In [5]:
def example_1_simple_tool():
    print("\n" + "="*70)
    print("示例 1：获取当前时间工具")
    print("="*70)
    @tool
    def get_current_time() -> str:
        """获取当前时间"""
        from datetime import datetime
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print("\n工具名称：", get_current_time.name)
    print("\n工具描述：", get_current_time.description)
    print("\n工具参数：", get_current_time.args)

    result = get_current_time.invoke({})
    print("\n工具调用：", result)


# 2.天气工具

In [8]:
def example_2_tool_with_params():
    print("\n" + "="*70)
    print("示例 2：天气工具")
    print("="*70)

    print("\n查看天气工具的信息：")
    print(f"名称: {get_weather.name}")
    print(f"描述: {get_weather.description}")
    print(f"参数: {get_weather.args}")

    # 测试工具
    res1 = get_weather.invoke({"city": "上海"})
    print("\n测试结果1：", res1)

    res2 = get_weather.invoke({"city": "北京"})
    print("\n测试结果2：", res2)

# 3.计算器工具

In [11]:
def example_3_multiple_params():
    print("\n" + "="*70)
    print("示例 3：计算器工具")
    print("="*70)

    print("\n计算器工具信息：")
    print(f"名称: {calculator.name}")
    print(f"描述: {calculator.description}")

    # 测试不同运算
    print("\n测试计算：")
    tests = [
        {"operation": "add", "a": 10, "b": 5},
        {"operation": "multiply", "a": 7, "b": 8},
        {"operation": "divide", "a": 20, "b": 4}
    ]

    for test in tests:
        result = calculator.invoke(test)
        print(f"  {result}")

# 4.搜索工具

In [12]:
def example_4_optional_params():
    print("\n" + "="*70)
    print("示例 4：搜索工具")
    print("="*70)

    # 使用默认参数
    print("\n使用默认参数（返回3条结果）：")
    result1 = web_search.invoke({"query": "Python"})
    print(result1)

    # 指定参数
    print("\n指定返回2个结果：")
    result2 = web_search.invoke({"query": "LangChain", "num_results": 2})
    print(result2)

# 5.工具绑定模型

In [13]:
def example_5_bind_tools():
    print("\n" + "="*70)
    print("示例 5：工具绑定到模型")
    print("="*70)

    # 绑定工具到模型
    model_with_tools = model.bind_tools([get_weather, calculator])

    print("模型已绑定工具：")
    print("  - get_weather")
    print("  - calculator")

    # 调用模型（模型可以选择使用工具）
    print("\n测试：AI 是否会调用天气工具？")
    response = model_with_tools.invoke("北京今天天气怎么样？")

    # 检查模型是否要求调用工具
    if response.tool_calls:
        print(f"\n✅ AI 决定使用工具！")
        print(f"工具调用: {response.tool_calls}")
    else:
        print(f"\nℹ️ AI 直接回答（未使用工具）")
        print(f"回复: {response.content}")

    print("\n💡 下一步：")
    print("  在 05_simple_agent 中，我们将学习如何让 AI 自动执行工具")

# ============================================================================
# 示例 6：工具的最佳实践
# ============================================================================

In [14]:
def main():
    print("\n" + "="*70)
    print(" LangChain 1.0 - 自定义工具")
    print("="*70)

    try:
        example_1_simple_tool()
        input("\n按 Enter 继续...")

        example_2_tool_with_params()
        input("\n按 Enter 继续...")

        example_3_multiple_params()
        input("\n按 Enter 继续...")

        example_4_optional_params()
        input("\n按 Enter 继续...")

        example_5_bind_tools()

        print("\n" + "="*70)
        print(" 完成！")
        print("="*70)

    except KeyboardInterrupt:
        print("\n\n程序中断")
    except Exception as e:
        print(f"\n错误: {e}")
        import traceback
        traceback.print_exc()

In [17]:
if __name__ == "__main__":
    main()


 LangChain 1.0 - 自定义工具

工具名称： get_current_time

工具描述： 获取当前时间

工具参数： {}

工具调用： 2026-04-01 08:01:01

示例 2：天气工具

查看天气工具的信息：
名称: get_weather
描述: 获取指定城市的天气信息

参数:
    city: 城市名称，如"北京"、"上海"

返回:
    天气信息字符串
参数: {'city': {'title': 'City', 'type': 'string'}}

测试结果1： 多云，温度 18°C，有轻微雾霾

测试结果2： 晴天，温度 15°C，空气质量良好

示例 3：计算器工具

计算器工具信息：
名称: calculator
描述: 执行基本的数学计算

参数:
    operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除)
    a: 第一个数字
    b: 第二个数字

返回:
    计算结果字符串

测试计算：
  10.0 add 5.0 = 15.0
  7.0 multiply 8.0 = 56.0
  20.0 divide 4.0 = 5.0

示例 4：搜索工具

使用默认参数（返回3条结果）：
搜索 'Python' 找到 3 条结果：
1. Python官方网站 - https://www.python.org
2. Python教程 - 菜鸟教程
3. Python最佳实践 - Real Python

指定返回2个结果：
搜索 'LangChain' 找到 2 条结果：
1. LangChain官方文档
2. LangChain GitHub仓库

示例 5：工具绑定到模型
模型已绑定工具：
  - get_weather
  - calculator

测试：AI 是否会调用天气工具？

✅ AI 决定使用工具！
工具调用: [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_317383caac074ab8973b9a', 'type': 'tool_call'}]

💡 下一步：
  在 05_simple_agent 中，我们将